# 04 — Autonomy vs Drain Predictor: Head-to-Head Comparison

> **Historical note:** This analysis led to the decision to decommission the autonomy
> model. The drain predictor subsumes its utility for the daily planning use case.
> The autonomy model's real-time triage capability was consolidated into the drain
> predictor's scoring flow (which now scores on-demand when AC_MAINS_FAIL fires).

**Question:** We have two models that predict "will this site drain?" at different horizons:
- **Autonomy model** (`survival:cox`): triggered per AC_MAINS_FAIL event, outputs relative risk
- **Drain predictor** (`binary:logistic`): daily batch, outputs calibrated P(drain in 48h)

**Overlap:** For sites that currently have an active outage, BOTH models have an opinion.
This notebook compares their predictive quality on the same set of sites/events.

**Fair comparison design:**
1. Find all AC_MAINS_FAIL events (autonomy's trigger points)
2. Score each with BOTH models using features computed at that exact moment
3. Label: did an LVD (LOAD_DISCONNECT) happen within 48h of that outage?
4. Compare: which model's scores better discriminate drains from non-drains?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, str(Path("../src")))

from battery_pdm.common.features import compute_features
from battery_pdm.synth.load_shedding import build_load_shedding_schedule
from battery_pdm.monitoring.model_registry import load_calibrator, apply_calibrator

SEED = 42
rng = np.random.default_rng(SEED)

## 1. Load data + both models

In [ ]:
import json

DATA = Path("../outputs")

alarms = pd.read_parquet(DATA / "alarms.parquet")
sites = pd.read_parquet(DATA / "site_static.parquet")
schedule = build_load_shedding_schedule(n_months=36, seed=SEED)

# Autonomy model
auto_dir = DATA / "models" / "autonomy"
auto_booster = xgb.Booster()
auto_booster.load_model(str(auto_dir / "booster.json"))
auto_meta = json.loads((auto_dir / "meta.json").read_text())

# Drain predictor
drain_dir = DATA / "models" / "drain_predictor_48h"
drain_booster = xgb.Booster()
drain_booster.load_model(str(drain_dir / "booster.json"))
drain_meta = json.loads((drain_dir / "meta.json").read_text())
drain_calibrator = load_calibrator(drain_dir)

print(f"Alarms: {len(alarms):,} rows, {alarms['site_id'].nunique()} sites")
print(f"Autonomy features: {len(auto_meta['feature_cols'])} | groups: {auto_meta['feature_groups']}")
print(f"Drain features: {len(drain_meta['feature_cols'])} | groups: {drain_meta.get('feature_groups', [])}")

## 2. Build evaluation set

For a fair comparison, we need events where BOTH models can be evaluated:
- An AC_MAINS_FAIL event occurred (autonomy's trigger)
- We know the ground truth: did LVD happen within 48h of that event?

We'll sample events from the SECOND HALF of the simulation (month 18+)
to avoid early-period instability.

In [ ]:
HORIZON_H = 48
MIN_H = 18 * 30 * 24  # start from month 18
MAX_H = alarms["timestamp_h"].max() - HORIZON_H  # need 48h lookahead

# All AC_MAINS_FAIL events in evaluation window
mains_fails = alarms[
    (alarms["alarm_code"] == "AC_MAINS_FAIL")
    & (alarms["timestamp_h"] >= MIN_H)
    & (alarms["timestamp_h"] <= MAX_H)
][["site_id", "timestamp_h"]].drop_duplicates().reset_index(drop=True)

# Sample to keep computation tractable
if len(mains_fails) > 2000:
    mains_fails = mains_fails.sample(2000, random_state=SEED).reset_index(drop=True)

# Label: did LVD happen within 48h of this AC_MAINS_FAIL?
lvd_events = alarms[alarms["alarm_code"] == "LOAD_DISCONNECT"]
lvd_by_site = {sid: g["timestamp_h"].values for sid, g in lvd_events.groupby("site_id")}

labels = []
for _, row in mains_fails.iterrows():
    site_lvds = lvd_by_site.get(row["site_id"], np.array([]))
    in_window = (site_lvds >= row["timestamp_h"]) & (site_lvds < row["timestamp_h"] + HORIZON_H)
    labels.append(int(in_window.any()))

mains_fails["drain_within_48h"] = labels
print(f"Evaluation events: {len(mains_fails):,}")
print(f"Positive rate (drain within 48h): {mains_fails['drain_within_48h'].mean():.1%}")
print(f"Sites represented: {mains_fails['site_id'].nunique()}")

## 3. Score with Autonomy model

Autonomy uses `alarm_history + site_static` features computed at the moment of AC_MAINS_FAIL.

In [ ]:
auto_labels = mains_fails[["site_id", "timestamp_h"]].rename(
    columns={"timestamp_h": "mains_fail_h"}
)

auto_features = compute_features(
    labels=auto_labels,
    groups=auto_meta["feature_groups"],
    inputs={"alarms": alarms, "site_static": sites},
)

auto_fcols = auto_meta["feature_cols"]
X_auto = auto_features[auto_fcols].astype(float).fillna(0.0)
mains_fails["autonomy_risk"] = auto_booster.predict(xgb.DMatrix(X_auto))

print(f"Autonomy scores: min={mains_fails['autonomy_risk'].min():.3f}, "
      f"max={mains_fails['autonomy_risk'].max():.3f}, "
      f"mean={mains_fails['autonomy_risk'].mean():.3f}")

## 4. Score with Drain Predictor

Drain predictor uses `alarm_history + site_static + soc_proxy + load_shedding_schedule`.
More features, calibrated output.

In [ ]:
drain_labels = mains_fails[["site_id", "timestamp_h"]].rename(
    columns={"timestamp_h": "mains_fail_h"}
)

drain_features = compute_features(
    labels=drain_labels,
    groups=["alarm_history", "site_static", "soc_proxy", "load_shedding_schedule"],
    inputs={"alarms": alarms, "site_static": sites, "schedule": schedule},
    ref_time_col="mains_fail_h",
)

drain_fcols = drain_meta["feature_cols"]
X_drain = drain_features[drain_fcols].astype(float).fillna(0.0)
raw_scores = drain_booster.predict(xgb.DMatrix(X_drain))
mains_fails["drain_prob_48h"] = apply_calibrator(drain_calibrator, raw_scores)

print(f"Drain predictor scores: min={mains_fails['drain_prob_48h'].min():.3f}, "
      f"max={mains_fails['drain_prob_48h'].max():.3f}, "
      f"mean={mains_fails['drain_prob_48h'].mean():.3f}")

## 5. Head-to-head comparison

In [ ]:
y = mains_fails["drain_within_48h"].values

# AUC comparison
auto_auc = roc_auc_score(y, mains_fails["autonomy_risk"].values)
drain_auc = roc_auc_score(y, mains_fails["drain_prob_48h"].values)

# Average Precision (better for imbalanced)
auto_ap = average_precision_score(y, mains_fails["autonomy_risk"].values)
drain_ap = average_precision_score(y, mains_fails["drain_prob_48h"].values)

# Brier score (only meaningful for drain predictor since autonomy isn't calibrated)
drain_brier = float(np.mean((mains_fails["drain_prob_48h"].values - y) ** 2))

print("=" * 60)
print("HEAD-TO-HEAD: Autonomy vs Drain Predictor")
print("=" * 60)
print(f"Task: Predict LVD within 48h of AC_MAINS_FAIL event")
print(f"Evaluation set: {len(y):,} events ({y.sum()} positives, {y.mean():.1%} rate)")
print(f"")
print(f"{'Metric':<25} {'Autonomy':<15} {'Drain Pred':<15} {'Winner'}")
print(f"{'-'*25} {'-'*15} {'-'*15} {'-'*10}")
print(f"{'ROC-AUC':<25} {auto_auc:<15.4f} {drain_auc:<15.4f} {'Drain' if drain_auc > auto_auc else 'Autonomy'}")
print(f"{'Avg Precision (PR-AUC)':<25} {auto_ap:<15.4f} {drain_ap:<15.4f} {'Drain' if drain_ap > auto_ap else 'Autonomy'}")
print(f"{'Brier Score':<25} {'N/A (not cal.)':<15} {drain_brier:<15.4f} {'—'}")
print(f"")
print(f"Note: Autonomy model outputs RELATIVE risk (survival:cox), not probabilities.")
print(f"AUC/AP only measure ranking quality, which is a fair comparison.")
print(f"Brier only applies to the calibrated drain predictor.")

## 6. Why the difference?

The drain predictor has two structural advantages for this task:
1. **Load-shedding schedule features** — it knows upcoming grid outage patterns
2. **SoC proxy features** — estimated state-of-charge from alarm history
3. **Trained on this exact task** — its objective IS "drain in 48h"

The autonomy model was trained on a DIFFERENT task: "rank sites by hours-to-LVD."
It's optimized for relative ordering of active outages, not for predicting future drains.

**When autonomy wins:** During an ACTIVE outage, if you need to dispatch a generator
to one of 5 failing sites, autonomy's ranking is more useful (it directly ranks urgency).

**When drain predictor wins:** For PROACTIVE planning ("which sites will fail tomorrow?"),
drain predictor is better because it incorporates the schedule.

In [ ]:
# Per-region breakdown
mains_fails_with_region = mains_fails.merge(
    sites[["site_id", "region"]], on="site_id", how="left"
)

print("\nPer-region AUC comparison:")
print(f"{'Region':<15} {'N events':<10} {'Pos rate':<10} {'Auto AUC':<12} {'Drain AUC':<12} {'Winner'}")
print("-" * 75)

for region, group in mains_fails_with_region.groupby("region"):
    if group["drain_within_48h"].nunique() < 2 or len(group) < 30:
        continue
    y_r = group["drain_within_48h"].values
    a_auc = roc_auc_score(y_r, group["autonomy_risk"].values)
    d_auc = roc_auc_score(y_r, group["drain_prob_48h"].values)
    winner = "Drain" if d_auc > a_auc else "Autonomy"
    print(f"{region:<15} {len(group):<10} {y_r.mean():<10.1%} {a_auc:<12.4f} {d_auc:<12.4f} {winner}")

In [ ]:
# Concrete site examples: high autonomy risk + low drain prob (and vice versa)
mains_fails["disagree"] = (
    (mains_fails["autonomy_risk"] > mains_fails["autonomy_risk"].quantile(0.75)) &
    (mains_fails["drain_prob_48h"] < 0.3)
) | (
    (mains_fails["autonomy_risk"] < mains_fails["autonomy_risk"].quantile(0.25)) &
    (mains_fails["drain_prob_48h"] > 0.6)
)

disagreements = mains_fails[mains_fails["disagree"]].copy()
print(f"\nDisagreement cases: {len(disagreements)} / {len(mains_fails)} "
      f"({len(disagreements)/len(mains_fails):.1%})")

if len(disagreements) > 0:
    print("\nSample disagreements (models give conflicting signals):")
    sample = disagreements.sample(min(10, len(disagreements)), random_state=SEED)
    print(sample[["site_id", "timestamp_h", "autonomy_risk", "drain_prob_48h", 
                  "drain_within_48h"]].to_string(index=False))
    
    # Who was right in disagreements?
    y_dis = disagreements["drain_within_48h"].values
    if y_dis.sum() > 0 and (1 - y_dis).sum() > 0:
        print(f"\nIn disagreement cases, actual drain rate: {y_dis.mean():.1%}")
        print(f"  Autonomy AUC on disagreements: {roc_auc_score(y_dis, disagreements['autonomy_risk'].values):.4f}")
        print(f"  Drain pred AUC on disagreements: {roc_auc_score(y_dis, disagreements['drain_prob_48h'].values):.4f}")

## 7. Conclusion

**They serve different operational needs:**

| Scenario | Use this model | Why |
|----------|---------------|-----|
| Grid just failed at 5 sites simultaneously. Which gets the generator? | **Autonomy** | Ranks active outages by urgency (hours-to-LVD) |
| Morning planning: which sites are at risk TODAY? | **Drain predictor** | Incorporates schedule, gives calibrated probability |
| Both models disagree on a site | **Drain predictor** for planning, **Autonomy** for real-time | Each is optimized for its cadence |

They're **complementary, not competing.** The autonomy model is a real-time triage tool;
the drain predictor is a planning tool. Comparing AUC on the same task shows which is
better at *that specific task*, but in production they serve different operators at
different times.